# Signature Identification - train the "whose signature is it?" model
Run the cells **from top to bottom**. Runtime: `Runtime > Change runtime type > T4 GPU`.

**Before you start, prepare your data on your PC:**
```
raw_signatures/
  Albert_Einstein/   1.jpg 2.jpg ... (8-10 or more different signatures)
  Abdul_Kalam/       1.jpg 2.jpg ...
  Marie_Curie/       1.jpg 2.jpg ...
```
One folder per person (use `_` instead of spaces), then zip the `raw_signatures` folder to `raw_signatures.zip`.

Use only public figures (or people who gave you permission). Every image must be a *different* signature (different document or year).

## Step 1: Install

In [ ]:
!pip -q install ultralytics

## Step 2: Upload the project zip (`yolo11-detection-app.zip`)

In [ ]:
from google.colab import files
import os, shutil, zipfile

uploaded = files.upload()
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall("/content")

PROJECT = "/content/signature_detection_project"
if os.path.exists(PROJECT):
    shutil.rmtree(PROJECT)
os.rename("/content/yolo11-detection-app", PROJECT)
print(os.listdir(PROJECT))

## Step 3: Signature detector (`best.pt`)
It is used to crop the signature from every photo.

In [ ]:
dst = f"{PROJECT}/weights/best.pt"
os.makedirs(f"{PROJECT}/weights", exist_ok=True)
candidates = [
    "/content/runs/signature/weights/best.pt",
    "/content/runs/detect/signature/weights/best.pt",
    "/content/drive/MyDrive/signature_detection_project/best.pt",
]
found = next((c for c in candidates if os.path.exists(c)), None)
if found:
    shutil.copy(found, dst)
    print("Detector copied from:", found)
else:
    print("Please upload best.pt (the signature detector)")
    up = files.upload()
    shutil.move(next(iter(up)), dst)
print("Detector ready:", os.path.exists(dst))

## Step 4: Upload `raw_signatures.zip`

In [ ]:
up = files.upload()
raw_zip = next(iter(up))
shutil.rmtree("/content/raw", ignore_errors=True)
with zipfile.ZipFile(raw_zip) as z:
    z.extractall("/content/raw")

# go into the folder that directly contains the person folders
RAW = "/content/raw"
IMG = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
while True:
    entries = [e for e in os.listdir(RAW) if not e.startswith((".", "__"))]
    dirs = [e for e in entries if os.path.isdir(os.path.join(RAW, e))]
    has_images = any(e.lower().endswith(IMG) for e in entries)
    if len(dirs) == 1 and not has_images:
        RAW = os.path.join(RAW, dirs[0])
    else:
        break

print("Using:", RAW)
for d in sorted(dirs):
    n = len([f for f in os.listdir(os.path.join(RAW, d)) if f.lower().endswith(IMG)])
    print(f"  {d:30s} {n} images")

## Step 5: Crop the signatures and split into train / val

In [ ]:
os.chdir(PROJECT)
!python training/prepare_people_dataset.py --raw "{RAW}" --detector weights/best.pt --out /content/signature_people

Check the table above: every person needs **at least 4 images** (8+ is much better).
`no-box` = photos where the detector found no signature (the whole photo was used instead). If this number is high, crop those photos by hand.

## Step 6: Train the identity model

In [ ]:
!python training/train_people.py --data /content/signature_people --epochs 60

Look at **Top-1 accuracy**: the share of validation signatures where the first guess was the right person.
- 0.9 or more: very good
- 0.7 to 0.9: usable, more samples per person will help
- below 0.7: add more different signatures per person

In [ ]:
import glob
from IPython.display import Image, display
for p in glob.glob(f"{PROJECT}/**/confusion_matrix_normalized.png", recursive=True)[:1]:
    display(Image(p))

## Step 7: Download the model
Put `people_cls.pt` in the `weights/` folder next to `best.pt`.

In [ ]:
files.download(f"{PROJECT}/weights/people_cls.pt")

In [ ]:
# Optional: keep a copy in Google Drive
from google.colab import drive
drive.mount("/content/drive")
folder = "/content/drive/MyDrive/signature_detection_project"
os.makedirs(folder, exist_ok=True)
shutil.copy(f"{PROJECT}/weights/people_cls.pt", f"{folder}/people_cls.pt")
print("Saved to Drive:", folder)